In [1]:
import os
import sys
import datetime as dt
from pathlib import Path

import pandas as pd
import geopandas as gpd

# ------------------------------------------------------------
# Optional r5py JVM memory setting before importing r5py
# ------------------------------------------------------------
sys.argv.extend(["--max-memory", "80%"])

import r5py  
print(r5py.__version__)



1.1.2


In [2]:
# ============================================================
# 0. CONFIGURATION
# ============================================================

PROJECT_ROOT = Path(__file__).parent if "__file__" in dir() else Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUT_DIR   = PROJECT_ROOT / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Inputs 
# .pbf works better in r5py
OSM_PBF   = DATA_DIR / "luxembourg-260310.osm.pbf"
GTFS_ZIP  = DATA_DIR / "gtfs.zip"

# If your GTFS is unpacked instead of zipped
GTFS_DIR  = DATA_DIR / "gtfs"
STOPS_TXT = GTFS_DIR / "stops.txt"

# Output
OUT_CSV = OUTPUT_DIR / "pt_times.csv"

# CRS
CRS_GEO = "EPSG:4326"

# Representative weekday window from phase 1
ANALYSIS_DATE   = dt.date(2026, 2, 4)
DEPARTURE_START = dt.datetime(2026, 2, 4, 7, 0, 0)
DEPARTURE_END   = dt.datetime(2026, 2, 4, 9, 0, 0)

DEPARTURE_WINDOW = DEPARTURE_END - DEPARTURE_START

# Snap settings
SNAP_RADIUS_M = 1600  # r5py default radius for walking snap is 1600 m

# Remove self-pairs in output
DROP_SELF_PAIRS = True

print('All good')


All good


In [4]:
# ============================================================
# 1. CHECK INPUTS
# ============================================================

print("Checking inputs...")

for path in [OSM_PBF, GTFS_ZIP, STOPS_TXT]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required input not found: {path}")

print('All good')

Checking inputs...
All good


In [5]:
# ============================================================
# 2. LOAD GTFS STOPS
# ============================================================

print("\nLoading GTFS stops...")

stops = pd.read_csv(STOPS_TXT, dtype={"stop_id": str})

required_cols = ["stop_id", "stop_lat", "stop_lon"]
missing = [c for c in required_cols if c not in stops.columns]
if missing:
    raise ValueError(f"stops.txt is missing required columns: {missing}")


# keep all stops that have valid coordinates.
stops = stops.dropna(subset=["stop_lat", "stop_lon"]).copy()

# Build GeoDataFrame in WGS84, which r5py expects for input geometries
stops_gdf = gpd.GeoDataFrame(
    stops[["stop_id"]].copy(),
    geometry=gpd.points_from_xy(stops["stop_lon"], stops["stop_lat"]),
    crs=CRS_GEO,
).rename(columns={"stop_id": "id"})

# Ensure ids are strings
stops_gdf["id"] = stops_gdf["id"].astype(str)

# Drop duplicate stop ids if any
stops_gdf = stops_gdf.drop_duplicates(subset=["id"]).reset_index(drop=True)

print(f"  GTFS stops loaded: {len(stops_gdf)}")





Loading GTFS stops...
  GTFS stops loaded: 2789


In [6]:
# ============================================================
# 3. BUILD R5PY TRANSPORT NETWORK
# ============================================================

print("\nBuilding r5py transport network...")

transport_network = r5py.TransportNetwork(
    OSM_PBF,
    [GTFS_ZIP],
)

print("All good")





Building r5py transport network...


All good


In [7]:
# ============================================================
# 4. SNAP STOPS TO NETWORK
# ============================================================

print("\nSnapping GTFS stops to r5py network...")

snapped = transport_network.snap_to_network(
    stops_gdf.geometry,
    radius=SNAP_RADIUS_M,
    street_mode=r5py.TransportMode.WALK,
)

stops_gdf = stops_gdf.copy()
stops_gdf["geometry"] = snapped

# Remove empty geometries that could not be snapped
not_empty = ~stops_gdf.geometry.is_empty
num_unsnapped = int((~not_empty).sum())
if num_unsnapped > 0:
    print(f"{num_unsnapped} stops could not be snapped and will be dropped.")

stops_gdf = stops_gdf.loc[not_empty].reset_index(drop=True)

print(f"  Stops after snapping: {len(stops_gdf)}")





Snapping GTFS stops to r5py network...
189 stops could not be snapped and will be dropped.
  Stops after snapping: 2600


In [8]:
# ============================================================
# 5. COMPUTE STOP-TO-STOP PT MATRIX
# ============================================================
print("\n Computing transit travel time matrix...")
print(f"  Date/window: {DEPARTURE_START} to {DEPARTURE_END}")

travel_time_matrix = r5py.TravelTimeMatrix(
    transport_network,
    origins=stops_gdf,
    destinations=stops_gdf,
    transport_modes=[r5py.TransportMode.TRANSIT],
    departure=DEPARTURE_START,
    departure_time_window=DEPARTURE_WINDOW,
)

matrix = pd.DataFrame(travel_time_matrix).copy()

print(f" Columns: {list(matrix.columns)}")





[5] Computing transit travel time matrix...
  Date/window: 2026-02-04 07:00:00 to 2026-02-04 09:00:00
 Columns: ['from_id', 'to_id', 'travel_time']


In [10]:
# ============================================================
# 6. STANDARDISE OUTPUT COLUMNS
# ============================================================

print("\n Standardising output columns...")

# r5py docs show columns from_id, to_id, travel_time
expected_map = {
    "from_id": "stop_id_from",
    "to_id": "stop_id_to",
    "travel_time": "pt_time_min",
}

missing_cols = [c for c in expected_map if c not in matrix.columns]
if missing_cols:
    raise ValueError(
        f"Unexpected r5py output columns. Missing expected columns: {missing_cols}. "
        f"Available columns: {list(matrix.columns)}"
    )

pt_times = matrix.rename(columns=expected_map)[
    ["stop_id_from", "stop_id_to", "pt_time_min"]
].copy()

pt_times["stop_id_from"] = pt_times["stop_id_from"].astype(str)
pt_times["stop_id_to"] = pt_times["stop_id_to"].astype(str)

# travel_time is documented in minutes in the user guide examples.
# But we still coerce safely in case of odd dtypes.
if pd.api.types.is_timedelta64_dtype(pt_times["pt_time_min"]):
    pt_times["pt_time_min"] = pt_times["pt_time_min"].dt.total_seconds() / 60.0
else:
    pt_times["pt_time_min"] = pd.to_numeric(pt_times["pt_time_min"], errors="coerce")

# Clean
pt_times = pt_times.dropna(subset=["pt_time_min"]).copy()
pt_times = pt_times[pt_times["pt_time_min"] >= 0].copy()

if DROP_SELF_PAIRS:
    pt_times = pt_times[pt_times["stop_id_from"] != pt_times["stop_id_to"]].copy()

# In case duplicates appear for any reason, keep the median
pt_times = (
    pt_times.groupby(["stop_id_from", "stop_id_to"], as_index=False)["pt_time_min"]
    .median()
)

pt_times = pt_times.sort_values(
    ["stop_id_from", "stop_id_to"], kind="mergesort"
).reset_index(drop=True)

print(f"  Cleaned stop-pair rows: {len(pt_times)}")





[6] Standardising output columns...
  Cleaned stop-pair rows: 4964083


In [11]:
# ============================================================
# 7. SAVE
# ============================================================

print("\n Saving pt_times.csv...")

pt_times.to_csv(OUT_CSV, index=False)

print(f"  Saved: {OUT_CSV}")
print("\nDone.")


[7] Saving pt_times.csv...
  Saved: /Users/fgsumer/Desktop/thesis_code/Phase 2/data/pt_times.csv

Done.


In [12]:
# ============================================================
# 8. WALIDATION CHECKS
# ============================================================

print("Rows:", len(pt_times))
print("Columns:", pt_times.columns.tolist())

print("\nSample rows:")
print(pt_times.head())

print("\nBasic stats:")
print(pt_times["pt_time_min"].describe())

print("\nNegative times:", (pt_times["pt_time_min"] < 0).sum())
print("Zero times:", (pt_times["pt_time_min"] == 0).sum())
print("NaN times:", pt_times["pt_time_min"].isna().sum())

print("\nUnique stops (from):", pt_times["stop_id_from"].nunique())
print("Unique stops (to):", pt_times["stop_id_to"].nunique())

Rows: 4964083
Columns: ['stop_id_from', 'stop_id_to', 'pt_time_min']

Sample rows:
   stop_id_from    stop_id_to  pt_time_min
0  000110101001  000110101002         10.0
1  000110101001  000110101004          9.0
2  000110101001  000110101008          7.0
3  000110101001  000110101009          9.0
4  000110101001  000110101010          3.0

Basic stats:
count    4.964083e+06
mean     7.739811e+01
std      2.567545e+01
min      0.000000e+00
25%      5.900000e+01
50%      7.900000e+01
75%      9.900000e+01
max      1.190000e+02
Name: pt_time_min, dtype: float64

Negative times: 0
Zero times: 156
NaN times: 0

Unique stops (from): 2600
Unique stops (to): 2600


In [13]:
(pt_times["stop_id_from"] == pt_times["stop_id_to"]).sum()
# zero-minute times are present but none are same-stop pairs

np.int64(0)

In [14]:

# check whether the zero pairs are symmetric duplicates  
zero_rows = pt_times[pt_times["pt_time_min"] == 0].copy()
print(zero_rows.head(20))
print("Zero rows:", len(zero_rows))
print("Unique from:", zero_rows["stop_id_from"].nunique())
print("Unique to:", zero_rows["stop_id_to"].nunique())

# there are symmetric duplicates but
# 156 rows out of ~5 million is negligible (<0.003%).

        stop_id_from    stop_id_to  pt_time_min
5988    000110101009  000110101011          0.0
8872    000110101011  000110101009          0.0
10365   000110101013  000110101017          0.0
15914   000110101017  000110101013          0.0
48439   000110109002  000110109004          0.0
52061   000110109004  000110109002          0.0
137288  000110606001  000110606008          0.0
145053  000110606008  000110606001          0.0
301670  000120606001  000120606005          0.0
307482  000120606005  000120606001          0.0
442735  000130210002  000140201005          0.0
476715  000140201005  000130210002          0.0
508296  000140304001  000140304002          0.0
510427  000140304002  000140304001          0.0
672146  000140701001  000140701024          0.0
696879  000140701024  000140701001          0.0
723768  000140701052  000140701053          0.0
726321  000140701053  000140701052          0.0
804515  000141301004  000141301005          0.0
806987  000141301005  000141301004      